# Bronze to Silver — CineData Analytics
Limpeza, padronização, tradução e deduplicação das tabelas Bronze, seguindo as regras de
negócio da seção 1.3 do enunciado. Nenhuma tabela Bronze é alterada.

In [ ]:
import os
import sys

sys.path.append(os.path.abspath(".."))

from src.silver.avaliacoes import transformar_avaliacoes_usuarios
from src.silver.cotacao_dolar import obter_cotacao_mais_recente, transformar_cotacao_dolar
from src.silver.engajamento import transformar_metricas_engajamento
from src.silver.filmes import transformar_info_filmes
from src.silver.financeiro import transformar_financeiro_filmes
from src.silver.generos import transformar_generos
from src.silver.pessoas_empresas import unificar_pessoas_empresas

spark.sql("CREATE DATABASE IF NOT EXISTS silver")

In [ ]:
# 1) silver.tb_info_filmes
df_info = spark.table("bronze.tb_movies_info")
df_filmes = transformar_info_filmes(df_info)
df_filmes.write.format("delta").mode("overwrite").saveAsTable("silver.tb_info_filmes")
display(df_filmes)

In [ ]:
# 3) silver.tb_metricas_engajamento
df_metrics = spark.table("bronze.tb_movies_metrics")
df_engajamento = transformar_metricas_engajamento(df_metrics)
df_engajamento.write.format("delta").mode("overwrite").saveAsTable("silver.tb_metricas_engajamento")
display(df_engajamento)

In [ ]:
# 4) silver.tb_avaliacoes_usuarios
df_reviews = spark.table("bronze.tb_movies_reviews")
df_avaliacoes = transformar_avaliacoes_usuarios(df_reviews)
df_avaliacoes.write.format("delta").mode("overwrite").saveAsTable("silver.tb_avaliacoes_usuarios")
display(df_avaliacoes)

In [ ]:
# 5) silver.tb_generos e 6) silver.tb_pessoas_empresas (mesma origem: tb_credits_and_tags)
df_credits = spark.table("bronze.tb_credits_and_tags")

df_generos = transformar_generos(df_credits)
df_generos.write.format("delta").mode("overwrite").saveAsTable("silver.tb_generos")

df_pessoas_empresas = unificar_pessoas_empresas(df_credits)
df_pessoas_empresas.write.format("delta").mode("overwrite").saveAsTable("silver.tb_pessoas_empresas")

display(df_pessoas_empresas)

In [ ]:
# 7) silver.tb_cotacao_dolar (forward-fill) -- precisa rodar antes do financeiro,
# que depende da cotacao mais recente calculada aqui.
df_cotacao_bronze = spark.table("bronze.tb_cotacao_dolar")
data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

df_cotacao_silver = transformar_cotacao_dolar(df_cotacao_bronze, spark, data_inicio, data_fim)
df_cotacao_silver.write.format("delta").mode("overwrite").saveAsTable("silver.tb_cotacao_dolar")

cotacao_atual = obter_cotacao_mais_recente(df_cotacao_silver)
print(f"Cotação usada para conversão BRL: {cotacao_atual}")

In [ ]:
# 2) silver.tb_financeiro_filmes (depende da cotacao calculada na celula anterior)
df_financials = spark.table("bronze.tb_movies_financials")
df_financeiro = transformar_financeiro_filmes(df_financials, cotacao=cotacao_atual)
df_financeiro.write.format("delta").mode("overwrite").saveAsTable("silver.tb_financeiro_filmes")
display(df_financeiro)